<a href="https://colab.research.google.com/github/PraveenKumar-pk-star/DS-corrected-file/blob/main/spread_sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-community
!pip install -q pypdf
!pip install -q pydantic
!pip install -q openai


In [2]:
import os
import getpass

from google.colab import files

from pydantic import BaseModel

from langchain_community.document_loaders import PyPDFLoader

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

In [3]:
os.environ["OPENAI_API_KEY"] = getpass.getpass(
    "Enter OpenAI API Key: "
)

Enter OpenAI API Key: ··········


In [4]:

uploaded = files.upload()

Saving dummy_hotel_booking_confirmation.pdf to dummy_hotel_booking_confirmation (3).pdf


In [5]:
pdf_file_name = list(uploaded.keys())[0]

print("Uploaded File:", pdf_file_name)

Uploaded File: dummy_hotel_booking_confirmation (3).pdf


In [6]:

loader = PyPDFLoader(pdf_file_name)

documents = loader.load()

In [7]:
full_text = ""

for doc in documents:
    full_text += doc.page_content + "\n"


print("\n================ PDF TEXT ================\n")

print(full_text[:3000])   # print first 3000 characters



================ PDF TEXT ================

Hotel Booking Confirmation
Hotel: Grand Vista Hotel
Guest Name
Praveen
Confirmation ID
GVH608483
Check-In Date
15 May 2026
Check-Out Date
18 May 2026
Room Type
Deluxe King Room
This is a dummy hotel booking confirmation generated for testing purposes only.



In [8]:
class BookingDetails(BaseModel):

    guest_name: str
    hotel_name: str
    confirmation_id: str
    check_in_date: str
    check_out_date: str

In [9]:
parser = PydanticOutputParser(
    pydantic_object=BookingDetails
)



In [10]:
prompt = PromptTemplate(

    template="""

    You are an expert hotel booking data extractor.

    Extract the following details from the hotel booking PDF text.

    Required Fields:
    - guest_name
    - hotel_name
    - confirmation_id
    - check_in_date
    - check_out_date

    Rules:
    - Return only structured output
    - If field missing return null
    - Do not make up information

    {format_instructions}

    HOTEL BOOKING PDF TEXT:
    {text}

    """,

    input_variables=["text"],

    partial_variables={
        "format_instructions":
        parser.get_format_instructions()
    }
)




In [11]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [12]:
chain = prompt | llm | parser


In [13]:
result = chain.invoke({
    "text": full_text
})

In [14]:
print("\n================ EXTRACTED DATA ================\n")

print("Guest Name      :", result.guest_name)
print("Hotel Name      :", result.hotel_name)
print("Confirmation ID :", result.confirmation_id)
print("Check In Date   :", result.check_in_date)
print("Check Out Date  :", result.check_out_date)


================ EXTRACTED DATA ================

Guest Name      : Praveen
Hotel Name      : Grand Vista Hotel
Confirmation ID : GVH608483
Check In Date   : 15 May 2026
Check Out Date  : 18 May 2026


In [15]:
result_dict = result.dict()

print("\n================ JSON OUTPUT ================\n")

print(result_dict)


================ JSON OUTPUT ================

{'guest_name': 'Praveen', 'hotel_name': 'Grand Vista Hotel', 'confirmation_id': 'GVH608483', 'check_in_date': '15 May 2026', 'check_out_date': '18 May 2026'}


/tmp/ipykernel_50135/1852542960.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  result_dict = result.dict()


In [24]:
from google.colab import files

uploaded = files.upload()

Saving excelautomation-496518-39a695d8e8a8.json to excelautomation-496518-39a695d8e8a8.json


In [17]:
!pip install -q gspread oauth2client

In [25]:
import gspread

from oauth2client.service_account import (
    ServiceAccountCredentials
)

In [26]:
scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]

creds = ServiceAccountCredentials.from_json_keyfile_name(
    "excelautomation-496518-39a695d8e8a8.json",
    scope
)

client = gspread.authorize(creds)

In [27]:
sheet = client.open_by_url(
    "https://docs.google.com/spreadsheets/d/1PNtZvdpU-91QrZWcmnBYFXzZhsNhx5vJIhb-A7qopoc/edit?usp=sharing" # <-- REPLACE THIS WITH YOUR GOOGLE SHEET LINK
).sheet1

In [30]:
row = [
    result.guest_name,
    result.hotel_name,
    result.confirmation_id,
    result.check_in_date,
    result.check_out_date
]

sheet.append_row(row)
print("Data uploaded successfully!")

Data uploaded successfully!


Data uploaded successfully!
